# Explicit live-kernel tracking — 0.4.6 development

Run each cell in an existing IPython kernel with the development Apizr candidate installed. The SDK does not import scientific libraries or start a kernel. This small example uses ordinary Python; real workloads keep their own scientific dependencies.


In [ ]:
from pathlib import Path

from apizr import tracking

root = Path.cwd()
measurements = [0.7, 0.8, 0.9]

## Track a block

The in-memory input is explicitly unknown. A file can instead be identified with `tracking.log_input("training", "data.csv")`. Metrics and parameters are explicit observations. Re-executing the next cell publishes another Run.


In [ ]:
with tracking.session(root=root) as capture:
    tracking.log_input("measurements")
    tracking.log_parameter("threshold", 0.5)
    tracking.log_metric("mean", sum(measurements) / len(measurements))

print("Captured cell", capture.record.run.capture.cell.execution_count)

In [ ]:
@tracking.experiment(parameters=["threshold"], root=root)
def evaluate(threshold):
    score = sum(value >= threshold for value in measurements) / len(measurements)
    tracking.log_input("measurements")
    tracking.log_metric("score", score)
    return score

In [ ]:
first = evaluate(0.75)
print("First score", first)

In [ ]:
second = evaluate(0.85)
print("Second score", second)

## Shared local history

Find these Runs with `apizr experiment list --store .apizr/experiments/v1`, inspect them with `show` and compare two with `diff`. Saved-notebook evidence stays unknown unless a source file is explicitly supplied; kernel evidence does not prove an exported serving binding. Interrupting an open session publishes a cancelled Run and keeps the interruption visible.
